# models.ipynb (UN-only, Regression-only)

Scope for final project alignment:
- Models: OLS, Ridge, Lasso, Random Forest, Gradient Boosting
- One target: UN revenue growth (`y_t1_un`)
- Feature sets: `numeric_only`, `+tier1`, `+tier2`
- No Tier 3 embeddings


In [1]:
# Step 1: Load and merge feature tiers (Tier 1 + Tier 2 only)
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
if not (ROOT / 'data').exists():
    ROOT = ROOT.parent

t1 = pd.read_parquet(ROOT / 'data' / 'features' / 'tier1.parquet')
t2 = pd.read_parquet(ROOT / 'data' / 'features' / 'tier2_tfidf.parquet')

features = t1.merge(t2, on=['year', 'segment'])
print('Merged features shape:', features.shape)

tier1_cols = [c for c in t1.columns if c not in ['year', 'segment']]
tier2_cols = [c for c in t2.columns if c not in ['year', 'segment']]
numeric_cols = ['n_meetings', 'n_tokens', 'sentiment']

print('tier1 cols:', len(tier1_cols))
print('tier2 cols:', len(tier2_cols))
print('numeric cols:', numeric_cols)


Merged features shape: (26, 2012)
tier1 cols: 10
tier2 cols: 2000
numeric cols: ['n_meetings', 'n_tokens', 'sentiment']


In [2]:
# Step 2: Load and aggregate UN revenue (only)
un_path_candidates = [
    ROOT / 'data' / 'un_panel.parquet',
    ROOT / 'data' / 'interim' / 'un_panel.parquet',
]
un_path = next((p for p in un_path_candidates if p.exists()), None)
if un_path is None:
    raise FileNotFoundError('Could not find un_panel.parquet in expected locations.')

un = pd.read_parquet(un_path)
un_annual = (
    un.groupby('year')['revenue_usd']
      .sum()
      .reset_index()
      .rename(columns={'revenue_usd': 'un_revenue'})
)

print('UN source:', un_path)
print('UN annual shape:', un_annual.shape)
print(un_annual.head())


UN source: /Users/enricoalmadani/Desktop/UChicago/Classes/26_Spring_Quarter/BUSN_20800/BUSN-20800-Final-Project-/data/un_panel.parquet
UN annual shape: (14, 2)
   year    un_revenue
0  2011  2.381769e+10
1  2012  2.503346e+10
2  2013  2.754187e+10
3  2014  2.872140e+10
4  2015  2.770679e+10


In [3]:
# Step 3: Carve WDI controls inline
wdi_path_candidates = [
    ROOT / 'data' / 'raw' / 'WDICSV.zip',
    ROOT / 'data' / 'raw' / 'WDICSV.csv',
]
wdi_path = next((p for p in wdi_path_candidates if p.exists()), None)
if wdi_path is None:
    raise FileNotFoundError('Could not find WDICSV.zip or WDICSV.csv in data/raw/.')

wdi = pd.read_csv(wdi_path)
indicator_map = {
    'NY.GDP.MKTP.KD.ZG': 'gdp_growth',
    'DT.ODA.ALLD.CD':    'oda_received',
    'GC.XPN.TOTL.GD.ZS': 'gov_expenditure_pct_gdp',
    'SP.DYN.LE00.IN':    'life_expectancy',
}

wdi_f = wdi[wdi['Indicator Code'].isin(indicator_map)].copy()
year_cols = [str(y) for y in range(2000, 2025)]
wdi_long = wdi_f.melt(
    id_vars=['Indicator Code'],
    value_vars=year_cols,
    var_name='year',
    value_name='value'
)
wdi_long['year'] = wdi_long['year'].astype(int)
wdi_long['indicator'] = wdi_long['Indicator Code'].map(indicator_map)
wdi_global = wdi_long.groupby(['year', 'indicator'])['value'].mean().unstack().reset_index()
wdi_global.columns.name = None

print('WDI source:', wdi_path)
print('WDI global shape:', wdi_global.shape)
print(wdi_global.head())


WDI source: /Users/enricoalmadani/Desktop/UChicago/Classes/26_Spring_Quarter/BUSN_20800/BUSN-20800-Final-Project-/data/raw/WDICSV.csv
WDI global shape: (25, 5)
   year  gdp_growth  gov_expenditure_pct_gdp  life_expectancy  oda_received
0  2000    4.552903                25.568300        67.153440  2.706759e+09
1  2001    3.243652                25.569187        67.494878  2.865428e+09
2  2002    3.454601                25.643970        67.786214  3.283235e+09
3  2003    3.927817                25.186592        68.114663  3.926410e+09
4  2004    5.909948                24.552741        68.449531  4.372183e+09


In [4]:
# Step 4: Build the full modeling table (UN target only)
df = (features
      .merge(un_annual, on='year')
      .merge(wdi_global, on='year', how='left'))

df = df.sort_values('year').reset_index(drop=True)

# One regression target: UN revenue growth
df['un_revenue_lag'] = df['un_revenue'].shift(1)
df['y_t1_un'] = np.log(df['un_revenue'] / df['un_revenue_lag'])
df['y_t1_un'] = df['y_t1_un'].replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=['y_t1_un'])

print('Modeling table shape:', df.shape)
print('Years in data:', df['year'].tolist())
print(df[['year', 'un_revenue', 'y_t1_un']].head())


Modeling table shape: (13, 2019)
Years in data: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
   year    un_revenue   y_t1_un
1  2012  2.503346e+10  0.049785
2  2013  2.754187e+10  0.095494
3  2014  2.872140e+10  0.041935
4  2015  2.770679e+10 -0.035965
5  2016  2.932663e+10  0.056819


In [5]:
# Step 5: Time-based train/val/test split
train = df[df['year'] <= 2019]
val   = df[(df['year'] >= 2020) & (df['year'] <= 2021)]
test  = df[df['year'] >= 2022]

print(f'Train: {len(train)}, Val: {len(val)}, Test: {len(test)}')
print('Train years:', train['year'].tolist())
print('Val years:', val['year'].tolist())
print('Test years:', test['year'].tolist())


Train: 8, Val: 2, Test: 3
Train years: [2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019]
Val years: [2020, 2021]
Test years: [2022, 2023, 2024]


In [6]:
# Step 6: Define feature sets (numeric_only, +tier1, +tier2)
control_cols = ['gdp_growth', 'gov_expenditure_pct_gdp', 'life_expectancy', 'oda_received']
model_numeric_cols = numeric_cols + control_cols
tier1_non_numeric = [c for c in tier1_cols if c not in numeric_cols]

feature_sets = {
    'numeric_only': model_numeric_cols,
    '+tier1':       model_numeric_cols + tier1_non_numeric,
    '+tier2':       model_numeric_cols + tier1_non_numeric + tier2_cols,
}

target_col = 'y_t1_un'

missing_controls = [c for c in control_cols if c not in df.columns]
if missing_controls:
    print('WARNING: missing controls in df:', missing_controls)

print('Target variable:', target_col)
for name, cols in feature_sets.items():
    print(f'{name}: {len(cols)} columns (before existence filter)')


Target variable: y_t1_un
numeric_only: 7 columns (before existence filter)
+tier1: 14 columns (before existence filter)
+tier2: 2014 columns (before existence filter)


### Why Scaling Here?
`StandardScaler` is applied to OLS, Ridge, and Lasso so coefficient penalties are comparable across features with different magnitudes (especially important with TF-IDF and macro controls). Tree models (RF, GBM) are left unscaled because they are scale-invariant.


In [7]:
# Step 7: Run 5 regression models and save results
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

def count_nonzero_coef(model, tol=1e-8):
    if hasattr(model, 'named_steps'):
        estimator = model.named_steps.get('model', None)
    else:
        estimator = model
    coef = getattr(estimator, 'coef_', None) if estimator is not None else None
    if coef is None:
        return None
    return int((np.abs(np.asarray(coef)) > tol).sum())

results = []

for feat_name, feat_cols in feature_sets.items():
    cols = [c for c in feat_cols if c in df.columns]

    X_train = train[cols].fillna(0)
    X_test  = test[cols].fillna(0)
    y_train = train[target_col]
    y_test  = test[target_col]

    models = {
        'OLS': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
        'Ridge': Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=1.0))]),
        'Lasso': Pipeline([('scale', StandardScaler()), ('model', Lasso(alpha=0.01, max_iter=20000))]),
        'RF': RandomForestRegressor(n_estimators=200, max_depth=3, random_state=42),
        'GBM': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
    }

    for model_name, model in models.items():
        model.fit(X_train, y_train)
        preds = model.predict(X_test)

        results.append({
            'model': model_name,
            'features': feat_name,
            'target': target_col,
            'n_features_used': len(cols),
            'n_nonzero_coef': count_nonzero_coef(model),
            'rmse': float(np.sqrt(mean_squared_error(y_test, preds))),
            'mae':  float(np.mean(np.abs(y_test - preds))),
            'r2':   float(r2_score(y_test, preds)),
        })

results_df = pd.DataFrame(results)
out_dir = ROOT / 'output'
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'model_results.csv'
results_df.to_csv(out_path, index=False)

print('Saved model_results.csv:', out_path)
print('Rows:', results_df.shape[0], 'Cols:', results_df.shape[1])
print(results_df.sort_values(['features', 'rmse']).to_string(index=False))


Saved model_results.csv: /Users/enricoalmadani/Desktop/UChicago/Classes/26_Spring_Quarter/BUSN_20800/BUSN-20800-Final-Project-/output/model_results.csv
Rows: 15 Cols: 8
model     features  target  n_features_used  n_nonzero_coef     rmse      mae           r2
   RF       +tier1 y_t1_un               14             NaN 0.163594 0.142623    -0.276852
Lasso       +tier1 y_t1_un               14             4.0 0.163618 0.127518    -0.277230
  GBM       +tier1 y_t1_un               14             NaN 0.174361 0.149304    -0.450468
Ridge       +tier1 y_t1_un               14            14.0 2.139502 1.268035  -217.389778
  OLS       +tier1 y_t1_un               14            14.0 2.824459 1.660146  -379.607759
  OLS       +tier2 y_t1_un             2014          2008.0 0.094279 0.090805     0.575926
Lasso       +tier2 y_t1_un             2014             5.0 0.140849 0.114586     0.053508
   RF       +tier2 y_t1_un             2014             NaN 0.154771 0.134054    -0.142848
  GBM       

In [8]:
# Step 7.5: Compact comparison tables for reporting
ranked = results_df.sort_values('rmse', ascending=True).copy()
print('\nTop models overall (by RMSE):')
print(ranked.head(10).to_string(index=False))

print('\nRMSE by model x feature set:')
rmse_pivot = results_df.pivot_table(index='model', columns='features', values='rmse', aggfunc='first')
print(rmse_pivot.to_string())

print('\nR2 by model x feature set:')
r2_pivot = results_df.pivot_table(index='model', columns='features', values='r2', aggfunc='first')
print(r2_pivot.to_string())

print('\nTier uplift vs numeric_only (negative is better for RMSE):')
uplift_rows = []
for m in sorted(results_df['model'].unique()):
    sub = results_df[results_df['model'] == m].set_index('features')
    if 'numeric_only' not in sub.index:
        continue
    base_rmse = float(sub.loc['numeric_only', 'rmse'])
    for tier in ['+tier1', '+tier2']:
        if tier in sub.index:
            rmse = float(sub.loc[tier, 'rmse'])
            uplift_rows.append({'model': m, 'tier': tier, 'delta_rmse_vs_numeric': rmse - base_rmse})

uplift_df = pd.DataFrame(uplift_rows).sort_values(['model', 'tier'])
print(uplift_df.to_string(index=False))



Top models overall (by RMSE):
model     features  target  n_features_used  n_nonzero_coef     rmse      mae        r2
  OLS       +tier2 y_t1_un             2014          2008.0 0.094279 0.090805  0.575926
Lasso numeric_only y_t1_un                7             2.0 0.127318 0.084276  0.226627
Lasso       +tier2 y_t1_un             2014             5.0 0.140849 0.114586  0.053508
   RF       +tier2 y_t1_un             2014             NaN 0.154771 0.134054 -0.142848
  GBM       +tier2 y_t1_un             2014             NaN 0.155621 0.133719 -0.155436
   RF numeric_only y_t1_un                7             NaN 0.158121 0.137646 -0.192847
   RF       +tier1 y_t1_un               14             NaN 0.163594 0.142623 -0.276852
Lasso       +tier1 y_t1_un               14             4.0 0.163618 0.127518 -0.277230
  GBM numeric_only y_t1_un                7             NaN 0.169690 0.143168 -0.373784
Ridge       +tier2 y_t1_un             2014          2008.0 0.172045 0.146525 -0.412187



In [9]:
# Step 7.6: Inspect Lasso selected variables and coefficient interpretation
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Lasso

def fit_lasso_and_extract(feature_set_name='+tier2', alpha=0.01, top_n=30):
    cols = [c for c in feature_sets[feature_set_name] if c in df.columns]

    X_train = train[cols].fillna(0)
    X_test  = test[cols].fillna(0)
    y_train = train[target_col]
    y_test  = test[target_col]

    pipe = Pipeline([
        ('scale', StandardScaler()),
        ('model', Lasso(alpha=alpha, max_iter=20000)),
    ])
    pipe.fit(X_train, y_train)

    y_pred = pipe.predict(X_test)
    rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
    mae = float(np.mean(np.abs(y_test - y_pred)))
    r2 = float(r2_score(y_test, y_pred))

    scaler = pipe.named_steps['scale']
    lasso = pipe.named_steps['model']

    coef_scaled = pd.Series(lasso.coef_, index=cols, name='coef_scaled')

    # Convert back to original feature units: beta_original = beta_scaled / std(x)
    # This helps explain effects in raw units.
    coef_original = pd.Series(lasso.coef_ / scaler.scale_, index=cols, name='coef_original_units')

    selected = coef_scaled[np.abs(coef_scaled) > 1e-8].copy()
    out = pd.concat([selected, coef_original[selected.index]], axis=1)
    out['abs_coef_scaled'] = out['coef_scaled'].abs()
    out = out.sort_values('abs_coef_scaled', ascending=False)

    print(f'Feature set: {feature_set_name}')
    print(f'alpha={alpha}')
    print(f'n_features_input={len(cols)} | n_features_selected={len(out)}')
    print(f'test RMSE={rmse:.6f} | MAE={mae:.6f} | R2={r2:.6f}')

    if len(out) == 0:
        print('No nonzero coefficients selected at this alpha.')
        return out

    print('\nTop selected variables:')
    print(out[['coef_scaled', 'coef_original_units']].head(top_n).to_string())

    # Quick category summary for interpretability
    def feat_type(name):
        if name.startswith('lex_'):
            return 'tier1_lexicon'
        if name.startswith('t_'):
            return 'tier2_tfidf'
        if name in ['n_meetings', 'n_tokens', 'sentiment']:
            return 'numeric_core'
        if name in ['gdp_growth', 'gov_expenditure_pct_gdp', 'life_expectancy', 'oda_received']:
            return 'macro_control'
        return 'other'

    type_counts = out.index.to_series().map(feat_type).value_counts()
    print('\nSelected-feature composition:')
    print(type_counts.to_string())

    return out

# Run this for each tier used in the paper
lasso_numeric = fit_lasso_and_extract('numeric_only', alpha=0.01, top_n=20)
print('\n' + '='*80 + '\n')
lasso_tier1 = fit_lasso_and_extract('+tier1', alpha=0.01, top_n=25)
print('\n' + '='*80 + '\n')
lasso_tier2 = fit_lasso_and_extract('+tier2', alpha=0.01, top_n=40)


Feature set: numeric_only
alpha=0.01
n_features_input=7 | n_features_selected=2
test RMSE=0.127318 | MAE=0.084276 | R2=0.226627

Top selected variables:
              coef_scaled  coef_original_units
gdp_growth       0.023168         9.917856e-02
oda_received     0.002893         5.372747e-12

Selected-feature composition:
macro_control    2


Feature set: +tier1
alpha=0.01
n_features_input=14 | n_features_selected=4
test RMSE=0.163618 | MAE=0.127518 | R2=-0.277230

Top selected variables:
                  coef_scaled  coef_original_units
gdp_growth           0.020577             0.088084
lex_humanitarian     0.009103             6.375617
lex_development     -0.003261            -0.877939
lex_gender           0.000314             0.706242

Selected-feature composition:
tier1_lexicon    3
macro_control    1


Feature set: +tier2
alpha=0.01
n_features_input=2014 | n_features_selected=5
test RMSE=0.140849 | MAE=0.114586 | R2=0.053508

Top selected variables:
                        coef_

### Extra Analysis: Rolling Expanding-Window Backtest
This tests year-to-year stability by repeatedly training on all years up to `t` and testing on `t+1`.


In [11]:
# Step 9: Rolling expanding-window backtest (same-year target y_t1_un)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

years = sorted(df['year'].unique().tolist())
# Require at least 6 train observations before first test year
min_train_size = 6

roll_rows = []
for i in range(min_train_size, len(years)):
    test_year = years[i]
    train_years = years[:i]

    tr = df[df['year'].isin(train_years)]
    te = df[df['year'] == test_year]

    for feat_name, feat_cols in feature_sets.items():
        cols = [c for c in feat_cols if c in df.columns]

        X_train = tr[cols].fillna(0)
        X_test  = te[cols].fillna(0)
        y_train = tr[target_col]
        y_test  = te[target_col]

        models = {
            'OLS': Pipeline([('scale', StandardScaler()), ('model', LinearRegression())]),
            'Ridge': Pipeline([('scale', StandardScaler()), ('model', Ridge(alpha=1.0))]),
            'Lasso': Pipeline([('scale', StandardScaler()), ('model', Lasso(alpha=0.01, max_iter=20000))]),
            'RF': RandomForestRegressor(n_estimators=200, max_depth=3, random_state=42),
            'GBM': GradientBoostingRegressor(n_estimators=200, max_depth=3, random_state=42),
        }

        for model_name, model in models.items():
            model.fit(X_train, y_train)
            pred = model.predict(X_test)[0]
            true = float(y_test.iloc[0])
            se = float((true - pred) ** 2)

            roll_rows.append({
                'analysis': 'rolling_backtest',
                'test_year': int(test_year),
                'train_start': int(train_years[0]),
                'train_end': int(train_years[-1]),
                'n_train': int(len(tr)),
                'model': model_name,
                'features': feat_name,
                'true': true,
                'pred': float(pred),
                'sq_error': se,
                'abs_error': float(abs(true - pred)),
            })

rolling_df = pd.DataFrame(roll_rows)

summary = (rolling_df
           .groupby(['model', 'features'], as_index=False)
           .agg(
               rmse_roll=('sq_error', lambda s: float(np.sqrt(np.mean(s)))),
               mae_roll=('abs_error', 'mean'),
               n_roll_tests=('test_year', 'count')
           )
           .sort_values('rmse_roll'))

print('Rolling backtest summary (lower rmse_roll is better):')
print(summary.to_string(index=False))

# Optional detailed pivot
print('\nRolling RMSE pivot (model x feature set):')
pivot = summary.pivot_table(index='model', columns='features', values='rmse_roll', aggfunc='first')
print(pivot.to_string())

roll_detail_path = ROOT / 'output' / 'rolling_backtest_detail.csv'
roll_summary_path = ROOT / 'output' / 'rolling_backtest_summary.csv'
rolling_df.to_csv(roll_detail_path, index=False)
summary.to_csv(roll_summary_path, index=False)
print('\nSaved:', roll_detail_path)
print('Saved:', roll_summary_path)


Rolling backtest summary (lower rmse_roll is better):
model     features  rmse_roll  mae_roll  n_roll_tests
  OLS       +tier2   0.080154  0.064863             7
   RF       +tier2   0.118178  0.078361             7
Ridge       +tier2   0.120508  0.088555             7
   RF       +tier1   0.125073  0.088738             7
   RF numeric_only   0.125243  0.088360             7
  GBM numeric_only   0.130260  0.092517             7
  GBM       +tier2   0.136705  0.093231             7
  GBM       +tier1   0.142083  0.094620             7
Lasso       +tier2   0.231382  0.147267             7
Lasso       +tier1   0.326489  0.193726             7
Lasso numeric_only   0.359369  0.215797             7
Ridge numeric_only   0.394588  0.265443             7
Ridge       +tier1   0.432207  0.296099             7
  OLS numeric_only   1.204303  0.811539             7
  OLS       +tier1   2.321108  1.036718             7

Rolling RMSE pivot (model x feature set):
features    +tier1    +tier2  numeric_o

### How to Interpret Lasso Coefficients
- `coef_scaled`: effect after z-scoring inputs (best for comparing relative importance across variables).
- `coef_original_units`: effect in raw feature units (better for substantive interpretation, but units differ across variables).
- Positive coefficient: higher feature value is associated with higher UN revenue growth (`y_t1_un`).
- Negative coefficient: higher feature value is associated with lower UN revenue growth.
- For TF-IDF (`t_*`) features, interpret as association with language usage intensity, not causal impact.
- Prefer discussing consistent signals (same sign across `numeric_only`, `+tier1`, `+tier2`) and keep claims directional due to tiny sample size.


### Interpretation Guide
- Baseline requirement is satisfied by OLS with `numeric_only`.
- Compare all five methods on out-of-sample RMSE/R².
- Use `n_nonzero_coef` to discuss Lasso sparsity and interpretability.
- Prefer claims that are robust across `numeric_only`, `+tier1`, and `+tier2` rather than single-cell wins.
- With very small test size, frame conclusions as directional evidence.
